# Minimum Diagnostic Toolkit for Paradoxical "Protective" Effects## When Diabetes Appears to Improve Survival: A Bias Detection Guide**Learning Objectives:**- Identify common biases in chronic disease cohort studies- Apply systematic diagnostics to detect immortal time, surveillance, and survivor biases- Implement appropriate corrections for detected biases**Key Principle:** > *Propensity score matching only handles measured baseline confounding that you correctly model. It cannot fix time zero misalignment, collider bias from selection, or surveillance artifacts.*

In [ ]:
# Import required librariesimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import statsfrom lifelines import KaplanMeierFitter, CoxPHFitterfrom lifelines.statistics import logrank_testfrom sklearn.linear_model import LogisticRegressionfrom sklearn.preprocessing import StandardScalerimport warningswarnings.filterwarnings('ignore')# Set styleplt.style.use('seaborn-v0_8-darkgrid')sns.set_palette("husl")np.random.seed(42)print("✓ Libraries loaded successfully")

---# Part 1: Conceptual Framework - DAGs and Bias MechanismsUnderstanding the causal structure is essential before any analysis.

In [ ]:
def create_dag_visualizations():    """Create DAG illustrations for different bias mechanisms"""    fig, axes = plt.subplots(2, 2, figsize=(16, 14))    fig.suptitle('Causal Structures: Truth vs Bias Mechanisms', fontsize=16, fontweight='bold')        # DAG 1: True causal structure    ax1 = axes[0, 0]    ax1.set_xlim(0, 10)    ax1.set_ylim(0, 10)    ax1.axis('off')    ax1.set_title('A. True Causal Structure\n(Diabetes → Mortality)', fontsize=14, fontweight='bold')        # Nodes    ax1.add_patch(plt.Circle((2, 7), 0.6, color='lightblue', ec='black', linewidth=2))    ax1.text(2, 7, 'Age', ha='center', va='center', fontsize=11, fontweight='bold')        ax1.add_patch(plt.Circle((2, 3), 0.6, color='lightgreen', ec='black', linewidth=2))    ax1.text(2, 3, 'Obesity', ha='center', va='center', fontsize=11, fontweight='bold')        ax1.add_patch(plt.Circle((5, 5), 0.7, color='#FF6B6B', ec='black', linewidth=3))    ax1.text(5, 5, 'Diabetes', ha='center', va='center', fontsize=11, fontweight='bold')        ax1.add_patch(plt.Circle((8, 5), 0.7, color='#FFE66D', ec='black', linewidth=3))    ax1.text(8, 5, 'Mortality', ha='center', va='center', fontsize=11, fontweight='bold')        # Arrows    ax1.annotate('', xy=(7.3, 5), xytext=(5.7, 5),                 arrowprops=dict(arrowstyle='->', lw=3, color='red'))    ax1.text(6.5, 5.5, 'HR > 1', ha='center', fontsize=10, color='red', fontweight='bold')        ax1.annotate('', xy=(4.7, 5.7), xytext=(2.5, 6.6),                 arrowprops=dict(arrowstyle='->', lw=2, color='gray'))    ax1.annotate('', xy=(7.5, 5.7), xytext=(2.5, 6.6),                 arrowprops=dict(arrowstyle='->', lw=2, color='gray'))    ax1.annotate('', xy=(4.7, 4.4), xytext=(2.5, 3.4),                 arrowprops=dict(arrowstyle='->', lw=2, color='gray'))    ax1.annotate('', xy=(7.5, 4.4), xytext=(2.5, 3.4),                 arrowprops=dict(arrowstyle='->', lw=2, color='gray'))        ax1.text(5, 1, 'TRUE EFFECT: Diabetes harmful or neutral',             ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat'))        # DAG 2: Collider bias    ax2 = axes[0, 1]    ax2.set_xlim(0, 10)    ax2.set_ylim(0, 10)    ax2.axis('off')    ax2.set_title('B. Collider Bias (Surveillance)\n(Conditioning on Healthcare Access)',                  fontsize=14, fontweight='bold', color='#FF6B35')        ax2.add_patch(plt.Circle((2, 7), 0.7, color='#FF6B6B', ec='black', linewidth=2))    ax2.text(2, 7, 'Diabetes', ha='center', va='center', fontsize=11, fontweight='bold')        ax2.add_patch(plt.Circle((8, 7), 0.7, color='lightgray', ec='black', linewidth=2))    ax2.text(8, 7, 'Frailty', ha='center', va='center', fontsize=11, fontweight='bold')        ax2.add_patch(plt.Rectangle((4.2, 4), 1.6, 1.2, color='orange', ec='black', linewidth=3))    ax2.text(5, 4.6, 'Healthcare\nUtilization', ha='center', va='center',             fontsize=10, fontweight='bold')    ax2.text(5, 3, 'COLLIDER!', ha='center', va='center', fontsize=12,             color='red', fontweight='bold')        ax2.add_patch(plt.Circle((5, 1), 0.6, color='#FFE66D', ec='black', linewidth=2))    ax2.text(5, 1, 'Detection', ha='center', va='center', fontsize=10, fontweight='bold')        ax2.annotate('', xy=(4.5, 5), xytext=(2.5, 6.5),                 arrowprops=dict(arrowstyle='->', lw=2.5, color='#FF6B35'))    ax2.annotate('', xy=(5.5, 5), xytext=(7.5, 6.5),                 arrowprops=dict(arrowstyle='->', lw=2.5, color='#FF6B35'))    ax2.annotate('', xy=(5, 1.6), xytext=(5, 4),                 arrowprops=dict(arrowstyle='->', lw=2, color='black'))        ax2.plot([2, 2, 8, 8], [6.3, 5, 5, 6.3], 'r--', lw=2.5, alpha=0.7)    ax2.text(5, 6, '← Spurious association →', ha='center',             fontsize=10, color='red', fontweight='bold')        ax2.text(5, 0.2, 'Conditioning on healthcare creates\nspurious negative association',             ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='#FFE6E6'))        # DAG 3: Immortal time bias    ax3 = axes[1, 0]    ax3.set_xlim(0, 10)    ax3.set_ylim(0, 10)    ax3.axis('off')    ax3.set_title('C. Immortal Time Bias\n(Misaligned Time Zero)',                  fontsize=14, fontweight='bold', color='#9B59B6')        # Timeline    ax3.plot([1, 9], [7, 7], 'k-', lw=3)    ax3.text(0.5, 7, 't=0', fontsize=11, fontweight='bold')    ax3.text(5, 7.5, 'Follow-up time →', fontsize=11, fontweight='bold')        # Exposed group    ax3.add_patch(plt.Rectangle((3, 5.5), 3, 0.8, color='#FF6B6B', alpha=0.3, ec='red', lw=2))    ax3.text(1.5, 5.9, 'Exposed:', fontsize=10, fontweight='bold')    ax3.plot([3, 3], [5.3, 6.5], 'g-', lw=3)    ax3.text(3, 5, 'Diagnosis/\nTreatment', ha='center', fontsize=9)    ax3.add_patch(plt.Rectangle((1, 5.5), 2, 0.8, color='lightgreen', alpha=0.5))    ax3.text(2, 5.9, 'Immortal\ntime', ha='center', fontsize=9, fontweight='bold', color='green')        # Unexposed group    ax3.add_patch(plt.Rectangle((1, 3.5), 5, 0.8, color='lightblue', alpha=0.3, ec='blue', lw=2))    ax3.text(1.5, 3.9, 'Unexposed:', fontsize=10, fontweight='bold')        ax3.text(5, 2.5, 'Exposed group guaranteed to survive\nuntil exposure → artificial protection',             ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='#E8DAEF'))        ax3.text(5, 1.2, 'FIX: Make exposure time-varying OR\nuse landmark analysis',             ha='center', fontsize=9, style='italic',             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))        # DAG 4: Prevalent user bias    ax4 = axes[1, 1]    ax4.set_xlim(0, 10)    ax4.set_ylim(0, 10)    ax4.axis('off')    ax4.set_title('D. Prevalent User Bias\n(Survivor Selection)',                  fontsize=14, fontweight='bold', color='#3498DB')        y_pos = 8    ax4.text(2, y_pos, 'All diabetes\npatients', ha='center', fontsize=10,             bbox=dict(boxstyle='round', facecolor='lightcoral'))        ax4.text(5, y_pos+0.5, '↓ Some die early', ha='center', fontsize=9, color='red')    ax4.text(5, y_pos, '↓ (before study entry)', ha='center', fontsize=8, color='red')        ax4.add_patch(plt.Rectangle((4, 5.5), 2, 1.5, color='lightgreen', ec='green', lw=3))    ax4.text(5, 6.2, 'Prevalent\nusers\n(survivors)', ha='center', fontsize=10, fontweight='bold')        ax4.text(5, 4.5, '↓', ha='center', fontsize=16, fontweight='bold')        ax4.text(5, 3.5, 'Selection of\ntolerant patients', ha='center', fontsize=10,            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))        ax4.text(5, 2, '"Depletion of susceptibles"\n→ Appears protective',             ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='#AED6F1'))        ax4.text(5, 0.5, 'FIX: Restrict to incident users\nwith washout period',             ha='center', fontsize=9, style='italic',            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))        plt.tight_layout()    return figfig_dag = create_dag_visualizations()plt.show()print("\n✓ DAG visualizations created")

---# Part 2: Timeline VisualizationsProper alignment of time zero is critical. Let's visualize correct vs incorrect approaches.

In [ ]:
def create_timeline_visualizations():    """Visualize time zero alignment scenarios"""    fig, axes = plt.subplots(3, 1, figsize=(16, 12))    fig.suptitle('Time Zero Alignment: Correct vs Incorrect Approaches',                 fontsize=16, fontweight='bold')        # Scenario 1: CORRECT    ax1 = axes[0]    ax1.set_xlim(-1, 11)    ax1.set_ylim(0, 6)    ax1.set_title('✓ CORRECT: Synchronized Time Zero', fontsize=14,                  fontweight='bold', color='green')    ax1.axis('off')        # Exposed patient    ax1.plot([0, 0], [4.5, 3.5], 'g-', lw=4)    ax1.text(-0.5, 4, 't₀', fontsize=12, fontweight='bold', color='green')    ax1.add_patch(plt.Rectangle((0, 3.5), 8, 0.6, color='#FF6B6B', alpha=0.6))    ax1.text(-0.5, 3.8, 'Exposed', fontsize=10, va='center', fontweight='bold')    ax1.scatter([0], [3.8], s=200, color='red', marker='*', zorder=10, edgecolor='black', lw=2)    ax1.text(0, 4.5, 'Diagnosis', fontsize=9, ha='center')    ax1.text(4, 3.8, 'Follow-up →', fontsize=10, va='center')        # Unexposed patient    ax1.plot([0, 0], [2.5, 1.5], 'g-', lw=4)    ax1.add_patch(plt.Rectangle((0, 1.5), 8, 0.6, color='lightblue', alpha=0.6))    ax1.text(-0.5, 1.8, 'Unexposed', fontsize=10, va='center', fontweight='bold')    ax1.text(4, 1.8, 'Follow-up →', fontsize=10, va='center')        ax1.text(4, 0.5, 'Both groups start follow-up simultaneously\nNo guaranteed survival time',             ha='center', fontsize=11,             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))        # Scenario 2: INCORRECT    ax2 = axes[1]    ax2.set_xlim(-1, 11)    ax2.set_ylim(0, 6)    ax2.set_title('✗ INCORRECT: Immortal Time Bias Introduced',                  fontsize=14, fontweight='bold', color='red')    ax2.axis('off')        ax2.plot([0, 0], [4.5, 3.5], 'r-', lw=4)    ax2.text(-0.5, 4, 't₀', fontsize=12, fontweight='bold', color='red')        # Immortal time    ax2.add_patch(plt.Rectangle((0, 3.5), 2.5, 0.6, color='lightgreen', alpha=0.8,                                 edgecolor='green', linewidth=3))    ax2.text(1.25, 3.8, 'IMMORTAL', fontsize=9, va='center', fontweight='bold', color='darkgreen')        # Exposed time    ax2.add_patch(plt.Rectangle((2.5, 3.5), 5.5, 0.6, color='#FF6B6B', alpha=0.6))    ax2.text(-0.5, 3.8, 'Exposed', fontsize=10, va='center', fontweight='bold')    ax2.scatter([2.5], [3.8], s=200, color='red', marker='*', zorder=10, edgecolor='black', lw=2)    ax2.text(2.5, 4.5, 'Diagnosis', fontsize=9, ha='center')    ax2.text(5.5, 3.8, 'Follow-up →', fontsize=10, va='center')        # Unexposed    ax2.plot([0, 0], [2.5, 1.5], 'r-', lw=4)    ax2.add_patch(plt.Rectangle((0, 1.5), 8, 0.6, color='lightblue', alpha=0.6))    ax2.text(-0.5, 1.8, 'Unexposed', fontsize=10, va='center', fontweight='bold')    ax2.text(4, 1.8, 'Follow-up (at risk from t₀) →', fontsize=10, va='center')        ax2.text(4, 0.5, 'Exposed group MUST survive to diagnosis\nUnexposed at risk from t₀ → Biased comparison',             ha='center', fontsize=11,             bbox=dict(boxstyle='round', facecolor='#FFE6E6', alpha=0.9))        # Scenario 3: CORRECTION    ax3 = axes[2]    ax3.set_xlim(-1, 11)    ax3.set_ylim(0, 6)    ax3.set_title('✓ CORRECTION: Landmark Analysis',                  fontsize=14, fontweight='bold', color='blue')    ax3.axis('off')        # Landmark time    ax3.plot([2.5, 2.5], [5, 1], 'b--', lw=3, alpha=0.7)    ax3.text(2.5, 5.3, 'Landmark\n(e.g., 6 months)', fontsize=10, ha='center',             fontweight='bold', color='blue')        # Exposed    ax3.add_patch(plt.Rectangle((2.5, 3.5), 5.5, 0.6, color='#FF6B6B', alpha=0.6))    ax3.text(2, 3.8, 'Exposed', fontsize=10, va='center', fontweight='bold')    ax3.scatter([1], [3.8], s=100, color='red', marker='*', zorder=10, alpha=0.3)    ax3.text(1, 4.3, 'Diagnosis', fontsize=8, ha='center', alpha=0.5)    ax3.text(5.5, 3.8, 'Follow-up →', fontsize=10, va='center')        # Unexposed    ax3.add_patch(plt.Rectangle((2.5, 1.5), 5.5, 0.6, color='lightblue', alpha=0.6))    ax3.text(2, 1.8, 'Unexposed', fontsize=10, va='center', fontweight='bold')    ax3.text(5.5, 1.8, 'Follow-up →', fontsize=10, va='center')        ax3.text(5.5, 0.5, 'Start follow-up at landmark time\nClassify exposure status at landmark\nExclude events before landmark',             ha='center', fontsize=11,             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))        plt.tight_layout()    return figfig_timeline = create_timeline_visualizations()plt.show()print("\n✓ Timeline visualizations created")

---# Part 3: Data SimulationWe'll simulate four scenarios to demonstrate each bias mechanism.

In [ ]:
def simulate_clean_data(n=5000, seed=42):    """Scenario A: Clean data with true harmful effect of diabetes"""    np.random.seed(seed)        # Baseline covariates    age = np.random.normal(65, 10, n)    age = np.clip(age, 40, 90)        bmi = np.random.normal(28, 5, n)    bmi = np.clip(bmi, 18, 45)        smoking = np.random.binomial(1, 0.2, n)        # Diabetes exposure    logit_diabetes = -3 + 0.03 * age + 0.1 * bmi + 0.5 * smoking    prob_diabetes = 1 / (1 + np.exp(-logit_diabetes))    diabetes = np.random.binomial(1, prob_diabetes)        # TRUE EFFECT: Diabetes increases mortality (HR = 1.5)    baseline_hazard = 0.01 + 0.001 * (age - 65) + 0.0005 * (bmi - 28) + 0.01 * smoking    hazard = baseline_hazard * np.exp(0.4 * diabetes)  # HR = exp(0.4) ≈ 1.5        # Generate survival times    u = np.random.uniform(0, 1, n)    time = -np.log(u) / hazard    time = np.clip(time, 0.1, 10)        event = (time < 10).astype(int)    time = np.minimum(time, 10)        df = pd.DataFrame({        'id': range(n),        'age': age,        'bmi': bmi,        'smoking': smoking,        'diabetes': diabetes,        'time': time,        'event': event    })        return dfdef simulate_immortal_time_bias(n=5000, seed=43):    """Scenario B: Immortal time bias - diagnosis occurs after cohort entry"""    np.random.seed(seed)    df = simulate_clean_data(n, seed)        # For exposed patients, diagnosis occurs during follow-up    diagnosis_time = np.where(df['diabetes'] == 1,                               np.random.uniform(0.5, 3, n),                              0)        # BIAS: If death before diagnosis, become "unexposed"    biased_diabetes = np.where((df['diabetes'] == 1) & (df['time'] > diagnosis_time), 1, 0)        df['diabetes_biased'] = biased_diabetes    df['diagnosis_time'] = diagnosis_time        return dfdef simulate_surveillance_bias(n=5000, seed=44):    """Scenario C: Surveillance/collider bias via differential healthcare"""    np.random.seed(seed)    df = simulate_clean_data(n, seed)        # Healthcare utilization is a collider    frailty = np.random.normal(0, 1, n)        logit_healthcare = -1 + 1.5 * df['diabetes'] + 1.5 * frailty    prob_healthcare = 1 / (1 + np.exp(-logit_healthcare))    high_healthcare = np.random.binomial(1, prob_healthcare)        clinic_visits = np.where(high_healthcare == 1,                            np.random.poisson(8, n),                            np.random.poisson(2, n))        hba1c_tests = np.where(df['diabetes'] == 1,                          np.random.poisson(4, n),                          np.random.poisson(0.5, n))        df['frailty'] = frailty    df['high_healthcare'] = high_healthcare    df['clinic_visits'] = clinic_visits    df['hba1c_tests'] = hba1c_tests    df['hazard_multiplier'] = np.exp(0.5 * frailty)        return dfdef simulate_survivor_bias(n=5000, seed=45):    """Scenario D: Prevalent user bias - only survivors included"""    np.random.seed(seed)    df = simulate_clean_data(n * 2, seed)        # Simulate 2-year pre-study period    u = np.random.uniform(0, 1, len(df))    baseline_hazard = 0.015 + 0.001 * (df['age'] - 65)    hazard_pre = baseline_hazard * np.exp(0.6 * df['diabetes'])    time_pre = -np.log(u) / hazard_pre        # Only include survivors    survived_2yr = time_pre > 2    df_prevalent = df[survived_2yr].copy().reset_index(drop=True)    df_prevalent = df_prevalent.head(n)    df_prevalent['prevalent_user'] = df_prevalent['diabetes']        return df_prevalent# Generate all scenariosprint("Simulating data for four scenarios...\n")df_clean = simulate_clean_data()print(f"✓ Scenario A (Clean): {len(df_clean)} patients")print(f"  Diabetes prevalence: {df_clean['diabetes'].mean():.2%}")print(f"  Event rate: {df_clean['event'].mean():.2%}\n")df_immortal = simulate_immortal_time_bias()print(f"✓ Scenario B (Immortal Time): {len(df_immortal)} patients")print(f"  Biased diabetes prevalence: {df_immortal['diabetes_biased'].mean():.2%}\n")df_surveillance = simulate_surveillance_bias()print(f"✓ Scenario C (Surveillance): {len(df_surveillance)} patients")print(f"  High healthcare utilization: {df_surveillance['high_healthcare'].mean():.2%}")print(f"  Mean clinic visits (diabetes): {df_surveillance[df_surveillance['diabetes']==1]['clinic_visits'].mean():.1f}")print(f"  Mean clinic visits (no diabetes): {df_surveillance[df_surveillance['diabetes']==0]['clinic_visits'].mean():.1f}\n")df_survivor = simulate_survivor_bias()print(f"✓ Scenario D (Survivor): {len(df_survivor)} patients")print(f"  Prevalent user rate: {df_survivor['prevalent_user'].mean():.2%}")print("\n" + "="*70)print("DATA SIMULATION COMPLETE")print("="*70)

---# Part 4: DIAGNOSTIC CHECKLISTNow we'll apply systematic diagnostics to detect biases.

## 4.1 Diagnostic #1: Definition VerificationFirst, verify basic study design elements.

In [ ]:
def diagnostic_1_definitions(df, scenario_name):    """Check incident vs prevalent, time zero alignment"""        print(f"\n{'='*70}")    print(f"DIAGNOSTIC #1: DEFINITION VERIFICATION - {scenario_name}")    print(f"{'='*70}\n")        checklist = [        "☐ Is diabetes INCIDENT (newly diagnosed) or PREVALENT (existing)?",        "☐ When is time zero (t₀) defined?",        "☐ Is exposure defined at t₀ or can it change over time?",        "☐ When does outcome follow-up begin?",        "☐ Are exposure and outcome timing properly aligned?"    ]        for item in checklist:        print(item)        print("\n" + "-"*70)    print("ANALYSIS FINDINGS:")    print("-"*70)        if 'diagnosis_time' in df.columns:        print("⚠️  WARNING: Exposure timing varies by patient")        print(f"   Diagnosis occurs 0.5-3 years after cohort entry")        print(f"   Mean diagnosis delay: {df[df['diabetes_biased']==1]['diagnosis_time'].mean():.2f} years")        print("   → IMMORTAL TIME BIAS LIKELY\n")        if 'prevalent_user' in df.columns:        print("⚠️  WARNING: Cohort includes prevalent users")        print("   Patients already had diabetes before study entry")        print("   → SURVIVOR BIAS LIKELY\n")        if 'diagnosis_time' not in df.columns and 'prevalent_user' not in df.columns:        print("✓ Exposure defined at baseline (time zero)")        print("✓ Follow-up begins at same time for all patients\n")        return# Run diagnostic on scenariosdiagnostic_1_definitions(df_clean, "SCENARIO A: CLEAN")diagnostic_1_definitions(df_immortal, "SCENARIO B: IMMORTAL TIME")diagnostic_1_definitions(df_survivor, "SCENARIO D: SURVIVOR")

## 4.2 Diagnostic #2: Negative Controls & Process VariablesCheck for differential healthcare utilization and testing patterns.

In [ ]:
def diagnostic_2_surveillance(df, scenario_name):    """Check healthcare utilization patterns"""        print(f"\n{'='*70}")    print(f"DIAGNOSTIC #2: SURVEILLANCE BIAS CHECK - {scenario_name}")    print(f"{'='*70}\n")        if 'clinic_visits' not in df.columns:        print("ℹ️  No healthcare utilization data available for this scenario\n")        return None        diabetes_yes = df[df['diabetes'] == 1]    diabetes_no = df[df['diabetes'] == 0]        print("HEALTHCARE UTILIZATION COMPARISON:")    print("-"*70)        metrics = {        'Clinic Visits': 'clinic_visits',        'HbA1c Tests': 'hba1c_tests'    }        results = []    for name, col in metrics.items():        mean_yes = diabetes_yes[col].mean()        mean_no = diabetes_no[col].mean()        ratio = mean_yes / mean_no if mean_no > 0 else np.inf                stat, pval = stats.mannwhitneyu(diabetes_yes[col], diabetes_no[col])                results.append({            'Metric': name,            'Diabetes': f"{mean_yes:.2f}",            'No Diabetes': f"{mean_no:.2f}",            'Ratio': f"{ratio:.2f}x",            'P-value': f"{pval:.4f}"        })        results_df = pd.DataFrame(results)    print(results_df.to_string(index=False))        print("\n" + "-"*70)    print("INTERPRETATION:")    print("-"*70)        if diabetes_yes['clinic_visits'].mean() / diabetes_no['clinic_visits'].mean() > 2:        print("⚠️  WARNING: Diabetes patients have >2x healthcare utilization")        print("   This creates differential surveillance")        print("   → COLLIDER BIAS LIKELY if conditioned on healthcare access\n")    else:        print("✓ Healthcare utilization similar between groups\n")        # Visualize    fig, axes = plt.subplots(1, 2, figsize=(14, 5))        axes[0].hist([diabetes_no['clinic_visits'], diabetes_yes['clinic_visits']],                 bins=20, alpha=0.6, label=['No Diabetes', 'Diabetes'], color=['lightblue', '#FF6B6B'])    axes[0].set_xlabel('Clinic Visits', fontsize=12)    axes[0].set_ylabel('Frequency', fontsize=12)    axes[0].set_title('Distribution of Clinic Visits', fontsize=13, fontweight='bold')    axes[0].legend()    axes[0].grid(alpha=0.3)        axes[1].hist([diabetes_no['hba1c_tests'], diabetes_yes['hba1c_tests']],                 bins=15, alpha=0.6, label=['No Diabetes', 'Diabetes'], color=['lightblue', '#FF6B6B'])    axes[1].set_xlabel('HbA1c Tests', fontsize=12)    axes[1].set_ylabel('Frequency', fontsize=12)    axes[1].set_title('Distribution of HbA1c Testing', fontsize=13, fontweight='bold')    axes[1].legend()    axes[1].grid(alpha=0.3)        plt.tight_layout()        return fig# Run diagnosticfig_surv = diagnostic_2_surveillance(df_surveillance, "SCENARIO C: SURVEILLANCE")if fig_surv:    plt.show()

## 4.3 Diagnostic #3: Survival Curve MorphologyExamine Kaplan-Meier curves for suspicious patterns.

In [ ]:
def diagnostic_3_survival_curves(scenarios_dict):    """Compare survival curve patterns across scenarios"""        print(f"\n{'='*70}")    print(f"DIAGNOSTIC #3: SURVIVAL CURVE MORPHOLOGY")    print(f"{'='*70}\n")        fig, axes = plt.subplots(2, 2, figsize=(16, 12))    axes = axes.flatten()        for idx, (scenario_name, df) in enumerate(scenarios_dict.items()):        ax = axes[idx]                diabetes_col = 'diabetes_biased' if 'diabetes_biased' in df.columns else 'diabetes'                kmf = KaplanMeierFitter()                # Diabetes group        diabetes_df = df[df[diabetes_col] == 1]        kmf.fit(diabetes_df['time'], diabetes_df['event'], label='Diabetes')        kmf.plot_survival_function(ax=ax, color='#FF6B6B', linewidth=2.5)                # No diabetes group        no_diabetes_df = df[df[diabetes_col] == 0]        kmf.fit(no_diabetes_df['time'], no_diabetes_df['event'], label='No Diabetes')        kmf.plot_survival_function(ax=ax, color='lightblue', linewidth=2.5)                # Log-rank test        results = logrank_test(            diabetes_df['time'], no_diabetes_df['time'],            diabetes_df['event'], no_diabetes_df['event']        )                # Calculate HR        cph = CoxPHFitter()        cph_df = df[['time', 'event', diabetes_col]].copy()        cph.fit(cph_df, duration_col='time', event_col='event')        hr = np.exp(cph.params_[diabetes_col])                ax.set_xlabel('Time (years)', fontsize=11)        ax.set_ylabel('Survival Probability', fontsize=11)        ax.set_title(f'{scenario_name}\nHR={hr:.2f}, p={results.p_value:.4f}',                     fontsize=12, fontweight='bold')        ax.legend(loc='lower left', fontsize=10)        ax.grid(alpha=0.3)        ax.set_ylim([0.4, 1.02])                # Interpretation        if hr < 0.9:            interpretation = "⚠️ PROTECTIVE effect detected\n(suspicious for bias)"            color = '#FFE6E6'        elif hr > 1.1:            interpretation = "✓ Harmful effect\n(as expected)"            color = 'lightgreen'        else:            interpretation = "Null effect"            color = 'lightyellow'                ax.text(0.98, 0.02, interpretation, transform=ax.transAxes,               fontsize=9, verticalalignment='bottom', horizontalalignment='right',               bbox=dict(boxstyle='round', facecolor=color, alpha=0.8))        plt.tight_layout()        print("INTERPRETATION GUIDE:")    print("-"*70)    print("• Early separation (immediate divergence):")    print("  → Check for immortal time bias or time zero misalignment")    print("\n• Late divergence/crossing curves:")    print("  → Consider frailty selection or competing risks")    print("\n• Protective effect (HR < 1):")    print("  → Highly suspicious if diabetes should be harmful")    print("  → Check all bias diagnostics immediately")    print("\n• Harmful effect (HR > 1):")    print("  → Consistent with expected biology")    print("  → Still verify no residual confounding\n")        return fig# Create scenarios dictionaryscenarios = {    'A. CLEAN (True Effect)': df_clean,    'B. IMMORTAL TIME BIAS': df_immortal,    'C. SURVEILLANCE BIAS': df_surveillance,    'D. SURVIVOR BIAS': df_survivor}fig_km = diagnostic_3_survival_curves(scenarios)plt.show()

---# Part 5: CORRECTION STRATEGIESNow we'll demonstrate how to fix each type of bias.

## 5.1 Correction for Immortal Time BiasTwo approaches: time-varying exposure and landmark analysis.

In [ ]:
def correct_immortal_time_bias(df):    """Demonstrate corrections for immortal time bias"""        print(f"\n{'='*70}")    print(f"CORRECTION #1: IMMORTAL TIME BIAS")    print(f"{'='*70}\n")        print("METHOD 1: Time-Varying Exposure")    print("-"*70)    print("Concept: Code exposure as time-varying covariate")    print("Implementation: Create person-time records with exposure status")    print("at each time point\n")        print("METHOD 2: Landmark Analysis (Implemented)")    print("-"*70)    print("Concept: Define landmark time (e.g., 1 year)")    print("Include only patients who survive to landmark")    print("Classify exposure at landmark, follow from landmark\n")        landmark_time = 1.0        # Keep only survivors to landmark    df_landmark = df[df['time'] > landmark_time].copy()    df_landmark['time_adjusted'] = df_landmark['time'] - landmark_time    df_landmark['diabetes_at_landmark'] = (        df_landmark['diagnosis_time'] <= landmark_time    ).astype(int)        print(f"Landmark time: {landmark_time} year")    print(f"Original cohort: {len(df)} patients")    print(f"After landmark restriction: {len(df_landmark)} patients")    print(f"Exposed at landmark: {df_landmark['diabetes_at_landmark'].sum()}\n")        # Compare biased vs corrected    fig, axes = plt.subplots(1, 2, figsize=(16, 6))        # BIASED    ax1 = axes[0]    kmf = KaplanMeierFitter()        diabetes_biased = df[df['diabetes_biased'] == 1]    kmf.fit(diabetes_biased['time'], diabetes_biased['event'], label='Diabetes (Biased)')    kmf.plot_survival_function(ax=ax1, color='red', linewidth=2.5, linestyle='--')        no_diabetes_biased = df[df['diabetes_biased'] == 0]    kmf.fit(no_diabetes_biased['time'], no_diabetes_biased['event'], label='No Diabetes')    kmf.plot_survival_function(ax=ax1, color='lightblue', linewidth=2.5)        cph = CoxPHFitter()    cph.fit(df[['time', 'event', 'diabetes_biased']], duration_col='time', event_col='event')    hr_biased = np.exp(cph.params_['diabetes_biased'])        ax1.set_xlabel('Time (years)', fontsize=12)    ax1.set_ylabel('Survival Probability', fontsize=12)    ax1.set_title(f'BIASED Analysis\nHR={hr_biased:.2f} (Protective - WRONG!)',                  fontsize=13, fontweight='bold', color='red')    ax1.legend(loc='lower left', fontsize=11)    ax1.grid(alpha=0.3)    ax1.axvline(landmark_time, color='green', linestyle=':', linewidth=2, alpha=0.7)    ax1.text(landmark_time+0.1, 0.5, 'Landmark', fontsize=10, color='green')        # CORRECTED    ax2 = axes[1]        diabetes_landmark = df_landmark[df_landmark['diabetes_at_landmark'] == 1]    kmf.fit(diabetes_landmark['time_adjusted'], diabetes_landmark['event'],            label='Diabetes (Corrected)')    kmf.plot_survival_function(ax=ax2, color='#FF6B6B', linewidth=2.5)        no_diabetes_landmark = df_landmark[df_landmark['diabetes_at_landmark'] == 0]    kmf.fit(no_diabetes_landmark['time_adjusted'], no_diabetes_landmark['event'],            label='No Diabetes')    kmf.plot_survival_function(ax=ax2, color='lightblue', linewidth=2.5)        cph.fit(df_landmark[['time_adjusted', 'event', 'diabetes_at_landmark']],            duration_col='time_adjusted', event_col='event')    hr_corrected = np.exp(cph.params_['diabetes_at_landmark'])        ax2.set_xlabel('Time since Landmark (years)', fontsize=12)    ax2.set_ylabel('Survival Probability', fontsize=12)    ax2.set_title(f'CORRECTED Landmark Analysis\nHR={hr_corrected:.2f} (Harmful - CORRECT!)',                  fontsize=13, fontweight='bold', color='green')    ax2.legend(loc='lower left', fontsize=11)    ax2.grid(alpha=0.3)        plt.tight_layout()        print("\nRESULTS COMPARISON:")    print("-"*70)    print(f"Biased HR (immortal time): {hr_biased:.3f}")    print(f"Corrected HR (landmark): {hr_corrected:.3f}")    print(f"\nBias magnitude: {abs(hr_biased - hr_corrected):.3f}")    print(f"Direction reversed: {(hr_biased < 1) != (hr_corrected < 1)}\n")        return fig, df_landmark# Apply correctionfig_correct1, df_immortal_corrected = correct_immortal_time_bias(df_immortal)plt.show()

## 5.2 Correction for Survivor BiasRestrict to incident users with washout period.

In [ ]:
def correct_survivor_bias():    """Demonstrate correction by using incident cohort"""        print(f"\n{'='*70}")    print(f"CORRECTION #2: SURVIVOR/PREVALENT USER BIAS")    print(f"{'='*70}\n")        print("METHOD: Restrict to Incident Users")    print("-"*70)    print("Concept: Include only newly diagnosed diabetes patients")    print("Implement washout period to ensure true incident cases")    print("Compare to all patients (no prior diabetes)\n")        # Generate incident cohort    df_incident = simulate_clean_data(n=5000, seed=50)        print("Cohort definitions:")    print(f"  Prevalent cohort: Includes existing diabetes patients")    print(f"  Incident cohort: New diabetes diagnoses only\n")        fig, axes = plt.subplots(1, 2, figsize=(16, 6))        # BIASED: Prevalent users    ax1 = axes[0]    kmf = KaplanMeierFitter()        diabetes_prev = df_survivor[df_survivor['diabetes'] == 1]    kmf.fit(diabetes_prev['time'], diabetes_prev['event'], label='Diabetes (Prevalent)')    kmf.plot_survival_function(ax=ax1, color='red', linewidth=2.5, linestyle='--')        no_diabetes_prev = df_survivor[df_survivor['diabetes'] == 0]    kmf.fit(no_diabetes_prev['time'], no_diabetes_prev['event'], label='No Diabetes')    kmf.plot_survival_function(ax=ax1, color='lightblue', linewidth=2.5)        cph = CoxPHFitter()    cph.fit(df_survivor[['time', 'event', 'diabetes']], duration_col='time', event_col='event')    hr_prevalent = np.exp(cph.params_['diabetes'])        ax1.set_xlabel('Time (years)', fontsize=12)    ax1.set_ylabel('Survival Probability', fontsize=12)    ax1.set_title(f'BIASED: Prevalent Users\nHR={hr_prevalent:.2f} (Appears protective)',                  fontsize=13, fontweight='bold', color='red')    ax1.legend(loc='lower left', fontsize=11)    ax1.grid(alpha=0.3)    ax1.text(0.5, 0.95, '⚠️ Survivor bias\n(tolerant patients)',             transform=ax1.transAxes, fontsize=10,             bbox=dict(boxstyle='round', facecolor='#FFE6E6', alpha=0.8),            verticalalignment='top')        # CORRECTED: Incident users    ax2 = axes[1]        diabetes_inc = df_incident[df_incident['diabetes'] == 1]    kmf.fit(diabetes_inc['time'], diabetes_inc['event'], label='Diabetes (Incident)')    kmf.plot_survival_function(ax=ax2, color='#FF6B6B', linewidth=2.5)        no_diabetes_inc = df_incident[df_incident['diabetes'] == 0]    kmf.fit(no_diabetes_inc['time'], no_diabetes_inc['event'], label='No Diabetes')    kmf.plot_survival_function(ax=ax2, color='lightblue', linewidth=2.5)        cph.fit(df_incident[['time', 'event', 'diabetes']], duration_col='time', event_col='event')    hr_incident = np.exp(cph.params_['diabetes'])        ax2.set_xlabel('Time (years)', fontsize=12)    ax2.set_ylabel('Survival Probability', fontsize=12)    ax2.set_title(f'CORRECTED: Incident Users\nHR={hr_incident:.2f} (True harmful effect)',                  fontsize=13, fontweight='bold', color='green')    ax2.legend(loc='lower left', fontsize=11)    ax2.grid(alpha=0.3)    ax2.text(0.5, 0.95, '✓ No survivor bias\n(new diagnoses)',             transform=ax2.transAxes, fontsize=10,             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8),            verticalalignment='top')        plt.tight_layout()        print("\nRESULTS COMPARISON:")    print("-"*70)    print(f"Biased HR (prevalent users): {hr_prevalent:.3f}")    print(f"Corrected HR (incident users): {hr_incident:.3f}")    print(f"\nBias magnitude: {abs(hr_prevalent - hr_incident):.3f}")    print(f"\nExplanation: Prevalent users are 'survivors' who tolerated diabetes")    print("Susceptible patients already died before study entry")    print("This creates depletion of susceptibles bias\n")        return figfig_correct2 = correct_survivor_bias()plt.show()

---# Part 6: Summary and Key Takeaways## The Fundamental Limitation of Propensity Score Matching**Critical Principle:**> *Propensity score matching only handles measured baseline confounding that you correctly model. It cannot automatically fix time zero misalignment, collider bias from selection, or surveillance artifacts created through the healthcare process.*## Why Chronic Diseases Create "Protective" IllusionsDiabetes and similar conditions are especially vulnerable to appearing "protective" due to:1. **Medical Contact Process**: More frequent visits create differential surveillance2. **Gradual Onset**: Makes time zero definition ambiguous3. **Prevalent Cases**: Long survival selects tolerant patients4. **Treatment Initiation**: Often delayed, creating guaranteed survival time

## Key Takeaways for Future Studies### ✓ Before Analysis1. **Define time zero clearly** - Must be identical for exposed and unexposed2. **Choose incident vs prevalent** - Prefer incident with washout period3. **Map causal structure** - Identify potential colliders (healthcare access)4. **Plan negative controls** - Select outcomes that should show null effects### ✓ During Analysis1. **Check survival curve morphology** - Early divergence suggests immortal time2. **Assess healthcare utilization** - Large differences indicate surveillance bias3. **Verify PS overlap** - Extreme weights signal positivity violations4. **Test negative controls** - Protective effects on injury death = red flag### ✓ Interpreting Results1. **Paradoxical findings demand explanation** - Protection from harmful exposure is suspicious2. **PS matching ≠ bias elimination** - Can't fix structural problems3. **Report limitations honestly** - Acknowledge biases you can't fix4. **Consider mechanism** - Does the effect make biological sense?### ✗ Common Mistakes to Avoid1. ❌ Including healthcare utilization in PS model without justification2. ❌ Using prevalent users without acknowledging survivor bias3. ❌ Ignoring time-varying exposure when diagnosis timing varies4. ❌ Accepting protective effects without rigorous bias assessment5. ❌ Claiming causality from observational data with known biases

## Quick Reference: Bias Detection & Correction| Bias Type | Detection Signal | Correction Method | When to Use ||-----------|-----------------|-------------------|-------------|| **Immortal Time** | Early curve separation; Diagnosis after t₀ | Time-varying exposure; Landmark analysis | Exposure timing varies; Treatment initiation studies || **Survivor** | Prevalent users; Selection of tolerant patients | Incident cohort; Washout period | Chronic diseases; Long-standing conditions || **Surveillance** | High testing/visit differential (>2x); Negative control fails | Don't condition on healthcare; IPCW | Registry data; Claims databases || **Positivity** | Poor PS overlap; Extreme weights | Restrict to common support; Report conditional | Rare exposures; Highly selective treatments |

## Final Thought> "When diabetes appears protective for mortality, the most likely explanation is not that it actually extends life, but that we've inadvertently selected for survivors, guaranteed immortal time, or conditioned on the medical care process. Our analysis tools are powerful, but they cannot overcome fundamental design flaws in how we define cohorts and time."**The best approach:** Design studies to avoid these biases from the start, rather than trying to adjust them away.---# References & Further Reading1. Suissa S. Immortal time bias in pharmacoepidemiology. *Am J Epidemiol*. 2008;167(4):492-499.2. Hernán MA, Sauer BC, Hernández-Díaz S, Platt R, Shrier I. Specifying a target trial prevents immortal time bias and other self-inflicted injuries in observational analyses. *J Clin Epidemiol*. 2016;79:70-75.3. Hernán MA, Hernández-Díaz S, Robins JM. A structural approach to selection bias. *Epidemiology*. 2004;15(5):615-625.4. Ray WA. Evaluating medication effects outside of clinical trials: new-user designs. *Am J Epidemiol*. 2003;158(9):915-920.5. Petersen ML, Porter KE, Gruber S, Wang Y, van der Laan MJ. Diagnosing and responding to violations in the positivity assumption. *Stat Methods Med Res*. 2012;21(1):31-54.---## Notebook CompleteThis toolkit provides:- ✅ Conceptual frameworks (DAGs)- ✅ Simulated examples of each bias- ✅ Systematic diagnostic checklist- ✅ Correction strategies with code- ✅ Visual summaries- ✅ Practical guidance for future studiesUse this as a reference when analyzing cohort studies of chronic disease effects!